# Tutorial: Machine Learning with Text in scikit-learn

# Note from Donny

This combines: Kevin Markham's Text Learning Tutorial<br>
Some Information from Wikipedia<br>
And some stuff on TF-IDF

## Agenda

1. Model building in scikit-learn (refresher)
2. Representing text as numerical data
3. Reading a text-based dataset into pandas
4. Vectorizing our dataset
5. Building and evaluating a model
6. Comparing models
7. Performing Cross-Validation to select C
8. Tuning the vectorizer (discussion)
9. TF-IDF Vectorizer plus model

## Part 1: Model building in scikit-learn (refresher)

In [1]:
# load the iris dataset as an example
from sklearn.datasets import load_iris
iris = load_iris()

In [2]:
# store the feature matrix (X) and response vector (y)
X = iris.data
y = iris.target

**"Features"** are also known as predictors, inputs, or attributes. The **"response"** is also known as the target, label, or output.

In [3]:
# check the shapes of X and y
print(X.shape)
print(y.shape)

(150, 4)
(150,)


**"Observations"** are also known as samples, instances, or records.

In [4]:
# examine the first 5 rows of the feature matrix (including the feature names)
import pandas as pd
pd.DataFrame(X, columns=iris.feature_names).head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [5]:
# examine the response vector
print(y)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2]


In order to **build a model**, the features must be **numeric**, and every observation must have the **same features in the same order**.

In [6]:
# import the class
from sklearn.neighbors import KNeighborsClassifier

# instantiate the model (with the default parameters)
knn = KNeighborsClassifier()

# fit the model with data (occurs in-place)
knn.fit(X, y)

KNeighborsClassifier()

In order to **make a prediction**, the new observation must have the **same features as the training observations**, both in number and meaning.

In [7]:
# predict the response for a new observation
knn.predict([[3, 5, 4, 2]])

array([1])

## Part 2: Representing text as numerical data

In [8]:
# example text for model training (SMS messages)
simple_train = ['call you tonight', 'Call me a cab', 'please call me... PLEASE!']

From the [scikit-learn documentation](http://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction):

> Text Analysis is a major application field for machine learning algorithms. However the raw data, a sequence of symbols cannot be fed directly to the algorithms themselves as most of them expect **numerical feature vectors with a fixed size** rather than the **raw text documents with variable length**.

We will use [CountVectorizer](http://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) to "convert text into a matrix of token counts":

In [9]:
# import and instantiate CountVectorizer (with the default parameters)
from sklearn.feature_extraction.text import CountVectorizer
vect = CountVectorizer()

In [10]:
# learn the 'vocabulary' of the training data (occurs in-place)
vect.fit(simple_train)

CountVectorizer()

In [11]:
simple_train

['call you tonight', 'Call me a cab', 'please call me... PLEASE!']

In [17]:
# examine the fitted vocabulary
vect.get_feature_names_out()

array(['cab', 'call', 'me', 'please', 'tonight', 'you'], dtype=object)

In [13]:
# transform training data into a 'document-term matrix'
simple_train_dtm = vect.transform(simple_train)
simple_train_dtm

<3x6 sparse matrix of type '<class 'numpy.int64'>'
	with 9 stored elements in Compressed Sparse Row format>

In [14]:
simple_train_dtm

<3x6 sparse matrix of type '<class 'numpy.int64'>'
	with 9 stored elements in Compressed Sparse Row format>

In [15]:
# convert sparse matrix to a dense matrix
simple_train_dtm.toarray()

array([[0, 1, 0, 0, 1, 1],
       [1, 1, 1, 0, 0, 0],
       [0, 1, 1, 2, 0, 0]], dtype=int64)

In [16]:
# examine the vocabulary and document-term matrix together
pd.DataFrame(simple_train_dtm.toarray(), columns=vect.get_feature_names_out())

,cab,call,me,please,tonight,you
0,0,1,0,0,1,1
1,1,1,1,0,0,0
2,0,1,1,2,0,0


From the [scikit-learn documentation](http://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction):

> In this scheme, features and samples are defined as follows:

> - Each individual token occurrence frequency (normalized or not) is treated as a **feature**.
> - The vector of all the token frequencies for a given document is considered a multivariate **sample**.

> A **corpus of documents** can thus be represented by a matrix with **one row per document** and **one column per token** (e.g. word) occurring in the corpus.

> We call **vectorization** the general process of turning a collection of text documents into numerical feature vectors. This specific strategy (tokenization, counting and normalization) is called the **Bag of Words** or "Bag of n-grams" representation. Documents are described by word occurrences while completely ignoring the relative position information of the words in the document.

In [18]:
# check the type of the document-term matrix
type(simple_train_dtm)

scipy.sparse._csr.csr_matrix

In [19]:
# examine the sparse matrix contents
print(simple_train_dtm)

  (0, 1)	1
  (0, 4)	1
  (0, 5)	1
  (1, 0)	1
  (1, 1)	1
  (1, 2)	1
  (2, 1)	1
  (2, 2)	1
  (2, 3)	2


From the [scikit-learn documentation](http://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction):

> As most documents will typically use a very small subset of the words used in the corpus, the resulting matrix will have **many feature values that are zeros** (typically more than 99% of them).

> For instance, a collection of 10,000 short text documents (such as emails) will use a vocabulary with a size in the order of 100,000 unique words in total while each document will use 100 to 1000 unique words individually.

> In order to be able to **store such a matrix in memory** but also to **speed up operations**, implementations will typically use a **sparse representation** such as the implementations available in the `scipy.sparse` package.

In [20]:
# example text for model testing
simple_test = ["please don't call me"]

In order to **make a prediction**, the new observation must have the **same features as the training observations**, both in number and meaning.

In [21]:
# transform testing data into a document-term matrix (using existing vocabulary)
simple_test_dtm = vect.transform(simple_test)
simple_test_dtm.toarray()

array([[0, 1, 1, 1, 0, 0]], dtype=int64)

In [22]:
# examine the vocabulary and document-term matrix together
pd.DataFrame(simple_test_dtm.toarray(), columns=vect.get_feature_names_out())

,cab,call,me,please,tonight,you
0,0,1,1,1,0,0


**Summary:**

- `vect.fit(train)` **learns the vocabulary** of the training data
- `vect.transform(train)` uses the **fitted vocabulary** to build a document-term matrix from the training data
- `vect.transform(valid)` uses the **fitted vocabulary** to build a document-term matrix from the validation data (and **ignores tokens** it hasn't seen before)

## Part 3: Reading a text-based dataset into pandas

Now you are going to do some things.

In [ ]:
# read file into pandas using a relative path sms.tsv
path = 'sms.tsv'
sms = pd.read_table(path, header=None, names=['label', 'message'])

Check the .head() as usual, did read_table work correctly? Do you need to set any parameters such as header or names?

We want the columns to have names <i>label</i> and <i>message</i>

In [ ]:
# Verify the read worked correctly
sms.head()

In [ ]:
# examine the shape
sms.shape

Examine the first 10 rows

In [ ]:
# examine the first 10 rows
sms.head(10)

In [ ]:
# examine the class distribution
sms.label.value_counts()

In [ ]:
# convert label to a numerical variable
sms['label_num'] = sms.label.map({'ham':0, 'spam':1})

In [ ]:
# check that the conversion worked
sms.head(10)

In [ ]:
# Identify what will be the response variable and what will be the set of features, the X and y
X = sms.message
y = sms.label_num
print(X.shape)
print(y.shape)

In [ ]:
# split X and y into training and validation sets - train_test_split with random_state=1138 so we all get the same
from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=1138, test_size=0.3)
print(X_train.shape)
print(X_valid.shape)
print(y_train.shape)
print(y_valid.shape)

## Part 4: Vectorizing our dataset

In [ ]:
# instantiate the vectorizer
vect = CountVectorizer()

### Notice

The Vectorizer (the vocabulary) is only built on the Training Set of Data. Can anyone tell me why?

The Vectorizer needs to be "fit" like a ML algorithm. Fit the <i>vect</i> with X_train

In [ ]:
# Fit the vectorizer with X_train
vect.fit(X_train)

You now need to create a dtm (document term matrix) like above by transforming the text into a matrix

In [ ]:
# Transform X_train into document-term matrix
X_train_dtm = vect.transform(X_train)

In [ ]:
# examine the document-term matrix
X_train_dtm

In [ ]:
import numpy as np

In [ ]:
for row in X_train_dtm.toarray():
     print(row)

In [ ]:
X_train_dtm.toarray()

The test data is transformed to the already fitted vocabulary. Words that were not in the original training set are ignored (our model would not know what to do with them). 

We could, of course, built this all as a pipeline using make_pipeline

In [ ]:
# transform testing data (using fitted vocabulary) into a document-term matrix
X_valid_dtm = vect.transform(X_valid)

## Part 5: Building and evaluating a model

We will use LogisticRegression to start with the default settings:

In [ ]:
# import and instantiate a LogisticRegression model
from sklearn.linear_model import LogisticRegression
nb = LogisticRegression(max_iter=1000)

Note: X_train_dtm is a sparse matrix, LogisticRegression understands sparse matrices, some models may have to do conversions.

In [ ]:
# train the model using X_train_dtm (timing it with an IPython "magic command")
%time nb.fit(X_train_dtm, y_train)

In [ ]:
# make class predictions for X_valid_dtm
y_pred_class = nb.predict(X_valid_dtm)

In [ ]:
# calculate accuracy of class predictions
from sklearn import metrics
metrics.accuracy_score(y_valid, y_pred_class)

In [ ]:
nb.score(X_valid_dtm, y_valid)

In [ ]:
# print the confusion matrix
metrics.confusion_matrix(y_valid, y_pred_class)

In [ ]:
# print the classification report
from sklearn.metrics import classification_report
print(classification_report(y_valid, y_pred_class))

In [ ]:
# print out all the "wrong" ones
X_valid[y_pred_class != y_valid]

In [ ]:
# print message text for the false positives (ham incorrectly classified as spam)
X_test[(y_pred_class==1) & (y_valid==0)] # or X_test[y_pred_class > y_test]

In [ ]:
# print message text for the false negatives (spam incorrectly classified as ham)
X_test[(y_pred_class==0) & (y_valid==1)]

In [ ]:
# example false negative
X_test[3132]

In [ ]:
# calculate predicted probabilities for X_test_dtm (poorly calibrated probabilities)
y_pred_prob = nb.predict_proba(X_valid_dtm)[:, 1]
y_pred_prob

In [ ]:
# calculate AUC - another metric for measure the performance of a classification system, it relies on probabilities. 
# I briefly mentioned it in lectures
metrics.roc_auc_score(y_valid, y_pred_prob)

## Part 6: Comparing models

We will compare LogisticRegression with a linear Support Vector Classifier and Random Forests:

Probability=True makes it take a bit longer to train but it will allow us to calculate the roc_auc score

In [ ]:
# import and instantiate a SVC model
from sklearn.svm import SVC
svcmodel = SVC(kernel='linear', probability=True)


In [ ]:
# train the model using X_train_dtm
%time svcmodel.fit(X_train_dtm, y_train)

In [ ]:
# make class predictions for X_valid_dtm
y_pred_class = svcmodel.predict(X_valid_dtm)

In [ ]:
# calculate predicted probabilities for X_valid_dtm (well calibrated)
y_pred_prob = svcmodel.predict_proba(X_valid_dtm)[:, 1]

In [ ]:
# calculate accuracy
metrics.accuracy_score(y_valid, y_pred_class)

In [ ]:
# calculate AUC
metrics.roc_auc_score(y_valid, y_pred_prob)

Now do Random Forests

In [ ]:
# import and instantiate a RandomForestClassifier model
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=1138)

In [ ]:
# train the model
%time rf.fit(X_train_dtm, y_train)

## Part 7: Cross Validation

Now you want to do some form of cross-validation with SVC, find the best 'C', use GridSearchCV or whatever method was your favourite. Find the best cross-validation score. You can set probability=False this time around and just use the accuracy score to compare

Same with Random Forest

In [ ]:
# Cross-validation with GridSearchCV for SVC
from sklearn.model_selection import GridSearchCV

# Define parameter grid
param_grid = {'C': [0.1, 1, 10, 100]}

# Create GridSearchCV object
grid_svc = GridSearchCV(SVC(kernel='linear', probability=True), 
                        param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Fit the grid search
%time grid_svc.fit(X_train_dtm, y_train)

In [ ]:
# Show best parameters and score
print('Best parameters:', grid_svc.best_params_)
print('Best cross-validation score:', grid_svc.best_score_)

In [ ]:
# Evaluate on validation set
y_pred = grid_svc.predict(X_valid_dtm)
print('Validation accuracy:', metrics.accuracy_score(y_valid, y_pred))
print('Validation AUC:', metrics.roc_auc_score(y_valid, grid_svc.predict_proba(X_valid_dtm)[:, 1]))

Now find the best 'C' with LogisticRegression using cross-validation, again find the best cross-validation score.

In [ ]:
# Cross-validation with GridSearchCV for LogisticRegression
from sklearn.model_selection import GridSearchCV

# Define parameter grid
param_grid = {'C': [0.01, 0.1, 1, 10, 100]}

# Create GridSearchCV object
grid_lr = GridSearchCV(LogisticRegression(max_iter=1000), 
                       param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Fit the grid search
%time grid_lr.fit(X_train_dtm, y_train)

In [ ]:
# Show best parameters and score
print('Best parameters:', grid_lr.best_params_)
print('Best cross-validation score:', grid_lr.best_score_)

In [ ]:
# Evaluate on validation set
y_pred = grid_lr.predict(X_valid_dtm)
print('Validation accuracy:', metrics.accuracy_score(y_valid, y_pred))
print('Validation AUC:', metrics.roc_auc_score(y_valid, grid_lr.predict_proba(X_valid_dtm)[:, 1]))

## Part 8: Tuning the vectorizer (discussion)

All of this can go into maybe a pipeline, keep the Vectorizer options as hyperparameters that could be chosen using cross-validation

Thus far, we have been using the default parameters of [CountVectorizer](http://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html):

In [ ]:
# show default parameters for CountVectorizer
vect

However, the vectorizer is worth tuning, just like a model is worth tuning! Here are a few parameters that you might want to tune:

- **stop_words:** string {'english'}, list, or None (default)
    - If 'english', a built-in stop word list for English is used.
    - If a list, that list is assumed to contain stop words, all of which will be removed from the resulting tokens.
    - If None, no stop words will be used.

In [ ]:
# remove English stop words
vect = CountVectorizer(stop_words='english')

In [ ]:
X_train_dtm = vect.fit_transform(X_train)
X_train_dtm

- **ngram_range:** tuple (min_n, max_n), default=(1, 1)
    - The lower and upper boundary of the range of n-values for different n-grams to be extracted.
    - All values of n such that min_n <= n <= max_n will be used.

In [ ]:
# include 1-grams and 2-grams
vect = CountVectorizer(ngram_range=(1, 2))

In [ ]:
X_train_dtm = vect.fit_transform(X_train)
X_train_dtm

- **max_df:** float in range [0.0, 1.0] or int, default=1.0
    - When building the vocabulary, ignore terms that have a document frequency strictly higher than the given threshold (corpus-specific stop words).
    - If float, the parameter represents a proportion of documents.
    - If integer, the parameter represents an absolute count.

In [ ]:
# ignore terms that appear in more than 50% of the documents
vect = CountVectorizer(max_df=0.5)

In [ ]:
X_train_dtm = vect.fit_transform(X_train)
X_train_dtm

- **min_df:** float in range [0.0, 1.0] or int, default=1
    - When building the vocabulary, ignore terms that have a document frequency strictly lower than the given threshold. (This value is also called "cut-off" in the literature.)
    - If float, the parameter represents a proportion of documents.
    - If integer, the parameter represents an absolute count.

In [ ]:
# only keep terms that appear in at least 2 documents
vect = CountVectorizer(min_df=2)

In [ ]:
X_train_dtm = vect.fit_transform(X_train)
X_train_dtm

**Guidelines for tuning CountVectorizer:**

- Use your knowledge of the **problem** and the **text**, and your understanding of the **tuning parameters**, to help you decide what parameters to tune and how to tune them.
- **Experiment**, and let the data tell you the best approach!
- We could use cross-validation to make our choices! 

We can also use a pipeline like

make_pipeline(CountVectorizer(), LogisticRegression())

we can do cross-validation over the CountVectorizer parameters then

# TF-IDF

The most commonly used technique is the tf-idf short for “term frequency-inverse document frequency”, which basically reflects how important a word is to a document (email) in a collection or corpus (our set of emails or documents).

## Term frequency

Suppose we have a set of English text documents and wish to rank which document is most relevant to the query, "the brown cow". A simple way to start out is by eliminating documents that do not contain all three words "the", "brown", and "cow", but this still leaves many documents. To further distinguish them, we might count the number of times each term occurs in each document; the number of times a term occurs in a document is called its term frequency. However, in the case where the length of documents varies greatly, adjustments are often made (see definition below). The first form of term weighting is due to Hans Peter Luhn (1957) which may be summarized as:

    The weight of a term that occurs in a document is simply proportional to the term frequency.

## Inverse document frequency
Because the term "the" is so common, term frequency will tend to incorrectly emphasize documents which happen to use the word "the" more frequently, without giving enough weight to the more meaningful terms "brown" and "cow". The term "the" is not a good keyword to distinguish relevant and non-relevant documents and terms, unlike the less-common words "brown" and "cow". Hence an inverse document frequency factor is incorporated which diminishes the weight of terms that occur very frequently in the document set and increases the weight of terms that occur rarely. 


## TF-IDF

The tf-idf is an statistic that increases with the number of times a word appears in the document, penalized by the number of documents in the corpus that contain the word.

Fortunately for us, Scikit-learn has a method that does just this (sklearn.feature_extraction.text.TfidfVectorizer). See the documentation [here](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
#TfidfVectorizer?

In [ ]:
tfidf_vectorizer = TfidfVectorizer(sublinear_tf=True, max_df=0.5, stop_words='english')

## Why these options?

TfidfVectorizer sets the vectorizer up. Here we change sublinear_tf to true, which replaces tf with 1 + log(tf). This addresses the issue that “twenty occurrences of a term in a document” does not represent “twenty times the significance of a single occurrence” [link](https://nlp.stanford.edu/IR-book/html/htmledition/sublinear-tf-scaling-1.html). Therefore, it reduces the importance of high frequency words (note that 1+log(1) = 1, while 1+log(20) = 2.3).

In [ ]:
corpus = [
    "This is my first email.",
    "I'm trying to learn machine learning.",
    "This is the second email",
    "Learning is fun"
]

In [ ]:
corpus_M = tfidf_vectorizer.fit_transform(corpus)

In [ ]:
print(corpus_M)

In [ ]:
vocabulary = tfidf_vectorizer.get_feature_names()

In [ ]:
print(vocabulary)

In [ ]:
pd.DataFrame(data=corpus_M.toarray(), columns=vocabulary)

In [ ]:
test = ["I’m also trying to learn python"]

In [ ]:
corpus_test = tfidf_vectorizer.transform(test)
pd.DataFrame(data=corpus_test.toarray(), columns=vocabulary)

## SMS Set

Use the SMS set to make a classifier using tfidf as our vectorizer instead of the bag of words

In [ ]:
# Fit and transform training data with TF-IDF
X_train_dtm = tfidf_vectorizer.fit_transform(X_train)

In [ ]:
X_train_dtm

In [ ]:
X_train_dtm.toarray()

In [ ]:
# Train a new logistic regression model
lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_dtm, y_train)

In [ ]:
# Transform validation data
X_valid_dtm = tfidf_vectorizer.transform(X_valid)

In [ ]:
X_valid_dtm

In [ ]:
# Make predictions
y_pred_class = lr_tfidf.predict(X_valid_dtm)

In [ ]:
# Calculate accuracy
metrics.accuracy_score(y_valid, y_pred_class)

In [ ]:
# Calculate score
lr_tfidf.score(X_valid_dtm, y_valid)

Take whatever was the best above (LogisticRegression, SVC, Random Forest) and see how it would compare with a TFIDF vectorizer

In [ ]:
# Compare best model with TF-IDF
# Let's use GridSearchCV with TF-IDF vectorizer
from sklearn.pipeline import Pipeline

# Create pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(sublinear_tf=True, max_df=0.5, stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000))
])

# Define parameter grid
param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [1, 2],
    'clf__C': [0.1, 1, 10]
}

# Grid search
grid_pipeline = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
%time grid_pipeline.fit(X_train, y_train)

print('Best parameters:', grid_pipeline.best_params_)
print('Best CV score:', grid_pipeline.best_score_)
print('Validation accuracy:', grid_pipeline.score(X_valid, y_valid))

In [ ]:
# Install tensorflow if needed
# !pip install tensorflow

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print('TensorFlow version:', tf.__version__)

## Prepare Data for Neural Network

Neural networks work with fixed-length sequences, so we'll:
1. Tokenize the text into integers
2. Pad sequences to the same length

In [ ]:
# Set parameters
vocab_size = 5000  # Maximum number of words to keep
max_length = 100   # Maximum length of sequences
embedding_dim = 32 # Dimension of word embeddings

# Create and fit tokenizer
tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# Convert texts to sequences of integers
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_valid_seq = tokenizer.texts_to_sequences(X_valid)

# Pad sequences to same length
X_train_padded = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
X_valid_padded = pad_sequences(X_valid_seq, maxlen=max_length, padding='post', truncating='post')

print('Training data shape:', X_train_padded.shape)
print('Validation data shape:', X_valid_padded.shape)

## Build Neural Network Model

We'll create a simple neural network with:
- Embedding layer: converts word indices to dense vectors
- Global Average Pooling: averages the embeddings
- Dense layers: for classification

In [ ]:
# Build the model
model = keras.Sequential([
    layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    layers.GlobalAveragePooling1D(),
    layers.Dense(24, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Display model architecture
model.summary()

## Train the Model

In [ ]:
# Train the model
history = model.fit(
    X_train_padded, 
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_valid_padded, y_valid),
    verbose=1
)

## Evaluate Performance

In [ ]:
# Evaluate on validation set
loss, accuracy = model.evaluate(X_valid_padded, y_valid)
print(f'\nValidation Accuracy: {accuracy:.4f}')
print(f'Validation Loss: {loss:.4f}')

In [ ]:
# Make predictions
y_pred_prob = model.predict(X_valid_padded)
y_pred_class = (y_pred_prob > 0.5).astype(int).flatten()

# Calculate metrics
print('Confusion Matrix:')
print(metrics.confusion_matrix(y_valid, y_pred_class))
print('\nClassification Report:')
print(metrics.classification_report(y_valid, y_pred_class))
print(f'\nAUC Score: {metrics.roc_auc_score(y_valid, y_pred_prob):.4f}')

## Visualize Training History

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(history.history['accuracy'], label='Training Accuracy')
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Model Accuracy')
ax1.legend()
ax1.grid(True)

# Loss
ax2.plot(history.history['loss'], label='Training Loss')
ax2.plot(history.history['val_loss'], label='Validation Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Model Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## Key Takeaways

1. **Traditional ML** (LogisticRegression, SVC) with TF-IDF or CountVectorizer:
   - Fast to train
   - Interpretable
   - Often performs well on text classification
   - Good baseline models

2. **Neural Networks**:
   - Can learn more complex patterns
   - Require more data and training time
   - Can benefit from pre-trained embeddings (Word2Vec, GloVe, BERT)
   - LSTM/Bidirectional LSTM: better for sequential patterns

3. **When to use what**:
   - Small datasets (<10k samples): Traditional ML often sufficient
   - Large datasets with complex patterns: Neural networks can excel
   - Production systems: Consider inference time and model size

4. **Next steps to explore**:
   - Pre-trained embeddings (Word2Vec, GloVe)
   - Transfer learning with BERT, RoBERTa, etc.
   - Attention mechanisms
   - Multi-task learning